In [1]:
from google.colab import drive  # Importing the library to mount Google Drive
drive.mount('/content/drive')  # Mounting Google Drive in Colab environment

Mounted at /content/drive


In [2]:
%%capture
!pip install keras_self_attention

In [ ]:
import pandas as pd

# File paths
train_df_file = "/content/drive/My Drive/NEW_DGA_DETECTOR/train_1M.csv"

train_df = pd.read_csv(train_df_file)

#train_df = train_df.rename(columns={"label": "Label"})


print(train_df)

                                         domain   family  label
0                               newmedialove.ru    legit  legit
1                                   bankesb.com    legit  legit
2                         sbjnvufhillsszger.net   fobber    dga
3                                  rpltm.online    legit  legit
4                               theblotsays.com    legit  legit
...                                         ...      ...    ...
1079995  c487bd0ebaf628e49016b23579812685.co.cc  bamital    dga
1079996   c21377bf084803b5d03161903bc5f643.info  bamital    dga
1079997                              yuncai.ltd    legit  legit
1079998                          movingscam.com    legit  legit
1079999                            aqaebif.info   pykspa    dga

[1080000 rows x 3 columns]


In [5]:
import datetime
import numpy as np
import pandas as pd

from keras.callbacks import ModelCheckpoint, History
from keras.models import Sequential
from keras.layers import Bidirectional, LSTM, Dense, Dropout, Embedding
from keras_self_attention import SeqSelfAttention, SeqWeightedAttention

## Charset and encoding/decoding functions
def encode(domain):
    # Convertir a minúsculas y filtrar caracteres no válidos
    domain = domain.lower()
    encoded = []
    for d in domain:
        if d in stoi:
            encoded.append(stoi[d])
        else:
            # Si el carácter no está en el charset, usar '*' como carácter desconocido
            encoded.append(stoi['*'])
    return encoded

def pad(l, amount=0, where='right', value=0):
    llen = len(l)
    if where == 'left':
        padded = [value]*(amount - llen) + l[:amount]
    if where == 'right':
        padded = l[:amount] + [value]*(amount - llen)
    return padded

# Charset expandido: incluye números, letras minúsculas, y caracteres comunes en dominios
charset = ['*'] + [chr(x) for x in range(0x30, 0x30+10)] + [chr(x) for x in range(0x61, 0x61+26)] + ['-', '_' ,'.']
stoi = {k:charset.index(k) for k in charset}
itos = {charset.index(k):k for k in charset}

print(f"Charset disponible: {''.join(charset)}")
print(f"Tamaño del vocabulario: {len(charset)}")

## Main parameters of the model
vocab_size = len(charset)
batch_size = 64
max_len = 64  # Maximum length for the domain names
embd_size = 128
lstm_size = 128
dense_size = 64
dropout = 0.5

## Data preparation function
def prepare_data(train_df):
    """
    Prepara los datos del dataframe para el entrenamiento
    train_df debe tener columnas 'domain' y 'label' (con valores 'dga' y 'notdga')
    """
    # Crear etiquetas binarias (1 para dga, 0 para notdga)
    df = train_df.copy()
    df['y'] = (df.label == 'dga').astype(int)

    # Codificar dominios
    df['encoded'] = df.domain.apply(encode)
    df['padded'] = df.encoded.apply(lambda x: pad(x, max_len, 'left'))

    # Convertir a arrays numpy
    X = np.array(list(df.padded.values))
    y = df['y'].values

    return X, y

## Callbacks para guardar el modelo y su historial de entrenamiento
def build_callbacks(save_path, monitor):
    checkpoint = ModelCheckpoint(filepath=save_path, monitor=monitor, verbose=1, save_best_only=True)
    history = History()
    callbacks = [checkpoint, history]
    return callbacks

# Crear callbacks
timestamp = str(datetime.datetime.now()).split(".")[0].replace(" ", "_")
labin_callbacks = build_callbacks(f'LABin_best_model_{timestamp}.keras', 'val_loss')

## LABin model definition - Binary classifier
LABin = Sequential()
LABin.add(Embedding(input_dim=vocab_size, output_dim=embd_size, input_length=max_len))
LABin.add(Bidirectional(LSTM(lstm_size, return_sequences=True), name="bilstm1"))
LABin.add(SeqSelfAttention(name="seqselfatt"))
LABin.add(Dropout(rate=dropout, name="drop1"))
LABin.add(Bidirectional(LSTM(lstm_size, return_sequences=True), name="bilstm2"))
LABin.add(SeqWeightedAttention(name="seqweigatt"))
LABin.add(Dropout(rate=dropout, name="drop2"))
LABin.add(Dense(dense_size, activation='relu', name="linear"))
LABin.add(Dropout(rate=dropout, name="drop3"))
LABin.add(Dense(1, activation='sigmoid', name="sigmoid"))
LABin.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])

# Mostrar resumen del modelo
LABin.summary()

## Función de entrenamiento
def train_labin(train_df, epochs=50, validation_split=0.2):
    """
    Entrena el modelo LABin con el dataframe proporcionado
    """
    print("Preparando datos...")
    X, y = prepare_data(train_df)

    print(f"Datos preparados: {X.shape[0]} muestras")
    print(f"Distribución de clases: DGA={np.sum(y)}, NotDGA={len(y)-np.sum(y)}")

    print("Iniciando entrenamiento...")
    history = LABin.fit(
        X, y,
        batch_size=batch_size,
        epochs=epochs,
        callbacks=labin_callbacks,
        validation_split=validation_split,
        verbose=1
    )

    return history

# Ejemplo de uso:
# Asumiendo que tienes tu dataframe 'train_df' con columnas 'domain' y 'label'
# history = train_labin(train_df, epochs=50)

## Función para visualizar resultados (opcional)
def plot_training_history(history):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('LABin Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('LABin Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f'LABin_training_history_{timestamp}.png')
    plt.show()

# Para usar después del entrenamiento:
# plot_training_history(history)

Charset disponible: *0123456789abcdefghijklmnopqrstuvwxyz-_.
Tamaño del vocabulario: 40


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm1 (Bidirectional)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ seqselfatt (SeqSelfAttention)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm2 (Bidirectional)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ seqweigatt                      │ ?                      │   0 (unbuilt) │
│ (SeqWeightedAttention)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sigmoid (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Ejemplo de uso:
# Asumiendo que tienes tu dataframe 'train_df' con columnas 'domain' y 'label'
history = train_labin(train_df, epochs=50)


Preparando datos...
Datos preparados: 1080000 muestras
Distribución de clases: DGA=540000, NotDGA=540000
Iniciando entrenamiento...
Epoch 1/50
13500/13500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8891 - loss: 0.2716
Epoch 1: val_loss improved from None to 0.14669, saving model to LABin_best_model_2026-03-23_12:55:03.keras

Epoch 1: finished saving model to LABin_best_model_2026-03-23_12:55:03.keras
13500/13500 ━━━━━━━━━━━━━━━━━━━━ 376s 27ms/step - accuracy: 0.9181 - loss: 0.2113 - val_accuracy: 0.9441 - val_loss: 0.1467
Epoch 2/50
13498/13500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9474 - loss: 0.1443
Epoch 2: val_loss improved from 0.14669 to 0.12402, saving model to LABin_best_model_2026-03-23_12:55:03.keras

Epoch 2: finished saving model to LABin_best_model_2026-03-23_12:55:03.keras
13500/13500 ━━━━━━━━━━━━━━━━━━━━ 349s 26ms/step - accuracy: 0.9498 - loss: 0.1386 - val_accuracy: 0.9538 - val_loss: 0.1240
Epoch 3/50
13500/13500 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accu

In [3]:
## FUNCIONES PARA CARGAR EL MODELO Y HACER PREDICCIONES

def load_trained_model(model_path):
    """
    Carga el modelo entrenado desde un archivo
    """
    from keras.models import load_model
    from keras_self_attention import SeqSelfAttention, SeqWeightedAttention

    # Cargar el modelo con las capas personalizadas
    custom_objects = {
        'SeqSelfAttention': SeqSelfAttention,
        'SeqWeightedAttention': SeqWeightedAttention
    }

    model = load_model(model_path, custom_objects=custom_objects)
    print(f"Modelo cargado desde: {model_path}")
    return model

def predict_single_domain(model, domain):
    """
    Predice si un dominio individual es DGA o no
    """
    # Preparar el dominio
    encoded = encode(domain)
    padded = pad(encoded, max_len, 'left')
    X = np.array([padded])  # Agregar dimensión batch

    # Hacer predicción
    prediction = model.predict(X, verbose=0)[0][0]

    # Interpretar resultado
    is_dga = prediction > 0.5
    confidence = prediction if is_dga else (1 - prediction)

    result = {
        'domain': domain,
        'prediction': 'DGA' if is_dga else 'LEGIT',
        'confidence': confidence,
        'raw_score': prediction
    }

    return result

def predict_domains_batch(model, domains_list):
    """
    Predice múltiples dominios a la vez
    """
    results = []

    # Preparar todos los dominios
    encoded_domains = [pad(encode(domain), max_len, 'left') for domain in domains_list]
    X = np.array(encoded_domains)

    # Hacer predicciones en lote
    predictions = model.predict(X, verbose=0)

    # Procesar resultados
    for i, domain in enumerate(domains_list):
        pred_score = predictions[i][0]
        is_dga = pred_score > 0.5
        confidence = pred_score if is_dga else (1 - pred_score)

        result = {
            'domain': domain,
            'prediction': 'DGA' if is_dga else 'LEGIT',
            'confidence': confidence,
            'raw_score': pred_score
        }
        results.append(result)

    return results

def evaluate_model_on_test(model, test_df):
    """
    Evalúa el modelo en un conjunto de test
    test_df debe tener columnas 'domain' y 'label'
    """
    print("Evaluando modelo en datos de test...")

    # Preparar datos de test
    X_test, y_test = prepare_data(test_df)

    # Hacer predicciones
    predictions = model.predict(X_test, verbose=0)
    y_pred = (predictions > 0.5).astype(int).flatten()

    # Calcular métricas
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm
    }

## EJEMPLOS DE USO:

"""
# 1. ENTRENAR EL MODELO
history = train_labin(train_df, epochs=50)

# 2. CARGAR UN MODELO YA ENTRENADO
# Cambia 'ruta_del_modelo.keras' por la ruta real donde guardaste tu modelo
loaded_model = load_trained_model('LABin_best_model_2025-05-30_15:22:09.keras')

# 3. PROBAR UN DOMINIO INDIVIDUAL
result = predict_single_domain(loaded_model, 'google.com')
print(f"Dominio: {result['domain']}")
print(f"Predicción: {result['prediction']}")
print(f"Confianza: {result['confidence']:.4f}")

# 4. PROBAR MÚLTIPLES DOMINIOS
test_domains = [
    'google.com',
    'facebook.com',
    'xkjhsdkjfhlksdjf.com',
    'qwerty123456.net',
    'amazon.com'
]

results = predict_domains_batch(loaded_model, test_domains)
for result in results:
    print(f"{result['domain']:<30} -> {result['prediction']:<5} (confianza: {result['confidence']:.4f})")

# 5. EVALUAR EN CONJUNTO DE TEST (si tienes un test_df)
# metrics = evaluate_model_on_test(loaded_model, test_df)
"""

'\n# 1. ENTRENAR EL MODELO\nhistory = train_labin(train_df, epochs=50)\n\n# 2. CARGAR UN MODELO YA ENTRENADO\n# Cambia \'ruta_del_modelo.keras\' por la ruta real donde guardaste tu modelo\nloaded_model = load_trained_model(\'LABin_best_model_2025-05-30_15:22:09.keras\')\n\n# 3. PROBAR UN DOMINIO INDIVIDUAL\nresult = predict_single_domain(loaded_model, \'google.com\')\nprint(f"Dominio: {result[\'domain\']}")\nprint(f"Predicción: {result[\'prediction\']}")\nprint(f"Confianza: {result[\'confidence\']:.4f}")\n\n# 4. PROBAR MÚLTIPLES DOMINIOS\ntest_domains = [\n    \'google.com\',\n    \'facebook.com\',\n    \'xkjhsdkjfhlksdjf.com\',\n    \'qwerty123456.net\',\n    \'amazon.com\'\n]\n\nresults = predict_domains_batch(loaded_model, test_domains)\nfor result in results:\n    print(f"{result[\'domain\']:<30} -> {result[\'prediction\']:<5} (confianza: {result[\'confidence\']:.4f})")\n\n# 5. EVALUAR EN CONJUNTO DE TEST (si tienes un test_df)\n# metrics = evaluate_model_on_test(loaded_model, test

In [6]:
# 2. CARGAR UN MODELO YA ENTRENADO
# Cambia 'ruta_del_modelo.keras' por la ruta real donde guardaste tu modelo
loaded_model = load_trained_model('/content/LABin_best_model_2026-03-23_12_55_03.keras')

# 3. PROBAR UN DOMINIO INDIVIDUAL
result = predict_single_domain(loaded_model, 'sadfdfdsfasds.com')
print(f"Dominio: {result['domain']}")
print(f"Predicción: {result['prediction']}")
print(f"Confianza: {result['confidence']:.4f}")

Modelo cargado desde: /content/LABin_best_model_2026-03-23_12_55_03.keras
Dominio: sadfdfdsfasds.com
Predicción: DGA
Confianza: 0.9343


In [7]:
import requests
import pandas as pd
import numpy as np
import time
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import sys
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
import re

families = [
    #'alureon.gz','bamital.gz', 'banjori.gz', 'bedep.gz', 'charbot.gz', 'chinad.gz',
    'conficker.gz', 'corebot.gz', 'cryptolocker.gz', 'deception.gz', 'dircrypt.gz',
    'dnschanger.gz', 'dyre.gz', 'emotet.gz', 'fobber.gz', 'gameover.gz', 'gozi.gz',
    'kraken.gz', 'locky.gz', 'manuelita.gz', 'matsnu.gz', 'monerominer.gz', 'murofet.gz',
    'murofetweekly.gz', 'mydoom.gz', 'necurs.gz', 'nymaim.gz', 'oderoor.gz', 'padcrypt.gz',
    'pitou.gz', 'proslikefan.gz', 'pushdo.gz', 'pykspa.gz', 'qadars.gz', 'qakbot.gz',
    'qsnatch.gz', 'ramdo.gz', 'ramnit.gz', 'ranbyus.gz', 'rovnix.gz', 'shiotob.gz',
    'simda.gz', 'sisron.gz', 'sphinx.gz', 'suppobox.gz', 'symmi.gz', 'tempedreve.gz',
    'tinba.gz', 'tinynuke.gz', 'vawtrak.gz', 'vidro.gz', 'virut.gz', 'zeus-newgoz.gz',
    'zloader.gz'
]

runs = 30

for family in families:
    print(family)
    dga = pd.read_csv(f'/content/drive/My Drive/Familias_Test/{family}', chunksize=50)
    legit = pd.read_csv('/content/drive/My Drive/Familias_Test/legit.gz', chunksize=50)
    dfs = []
    for run in range(runs):
        print(f'{run:2}/{runs}', end='\r')
        dfw = pd.concat([dga.get_chunk(), legit.get_chunk()])
        pred = []
        prob = []
        query_time = []
        results = []

        for domain_to_check in dfw.domain.values:
            st = time.time()

            result = predict_single_domain(loaded_model, domain_to_check)
            if result['prediction'] == "DGA":
                label_value = 1
            else:
                label_value = 0

            pred.append(label_value)
            query_time.append(time.time() - st)

        dfw['pred'] = pred
        # dfw['prob'] = prob  # Si tienes probabilidades, descomenta esta línea
        dfw['query_time'] = query_time
        dfw.to_csv(f'/content/drive/My Drive/results/results_Labin_{family}_{run}.csv.gz', index=False)


conficker.gz
corebot.gz
cryptolocker.gz
deception.gz
dircrypt.gz
dnschanger.gz
dyre.gz
emotet.gz
fobber.gz
gameover.gz
gozi.gz
kraken.gz
locky.gz
manuelita.gz
matsnu.gz
monerominer.gz
murofet.gz
murofetweekly.gz
mydoom.gz
necurs.gz
nymaim.gz
oderoor.gz
padcrypt.gz
pitou.gz
proslikefan.gz
pushdo.gz
pykspa.gz
qadars.gz
qakbot.gz
qsnatch.gz
ramdo.gz
ramnit.gz
ranbyus.gz
rovnix.gz
shiotob.gz
simda.gz
sisron.gz
sphinx.gz
suppobox.gz
symmi.gz
tempedreve.gz
tinba.gz
tinynuke.gz
vawtrak.gz
vidro.gz
virut.gz
zeus-newgoz.gz
zloader.gz


In [11]:
#"""
families = [
    'alureon.gz','bamital.gz', 'banjori.gz', 'bedep.gz', 'charbot.gz', 'chinad.gz',
    'conficker.gz', 'corebot.gz', 'cryptolocker.gz', 'deception.gz', 'dircrypt.gz',
    'dnschanger.gz', 'dyre.gz', 'emotet.gz', 'fobber.gz', 'gameover.gz', 'gozi.gz',
    'kraken.gz', 'locky.gz', 'manuelita.gz', 'matsnu.gz', 'monerominer.gz', 'murofet.gz',
    'murofetweekly.gz', 'mydoom.gz', 'necurs.gz', 'nymaim.gz', 'oderoor.gz', 'padcrypt.gz',
    'pitou.gz', 'proslikefan.gz', 'pushdo.gz', 'pykspa.gz', 'qadars.gz', 'qakbot.gz',
    'qsnatch.gz', 'ramdo.gz', 'ramnit.gz', 'ranbyus.gz', 'rovnix.gz', 'shiotob.gz',
    'simda.gz', 'sisron.gz', 'sphinx.gz', 'suppobox.gz', 'symmi.gz', 'tempedreve.gz',
    'tinba.gz', 'tinynuke.gz', 'vawtrak.gz', 'vidro.gz', 'virut.gz', 'zeus-newgoz.gz',
    'zloader.gz'
]
#"""
# Listas para métricas globales
all_acc, all_pre, all_rec, all_f1 = [], [], [], []
all_fpr, all_tpr, all_qt, all_qts = [], [], [], []
total_unknowns_global = 0

def fpr_tpr(y, ypred):
    tn, fp, fn, tp = confusion_matrix(y, ypred).ravel()
    fpr = fp / (fp + tn)  # False Positive Rate
    tpr = tp / (tp + fn)  # True Positive Rate (Recall)
    return fpr, tpr

for family in families:
    acc = []
    pre = []
    rec = []
    f1 = []
    fpr = []
    tpr = []
    qt = []
    qts = []
    for run in range(runs):
        df = pd.read_csv(f'/content/drive/My Drive/results/results_Labin_{family}_{run}.csv.gz')
        y = (df.label == 'dga').astype(int)
        ypred = df.pred
        acc.append(accuracy_score(y, ypred))
        pre.append(precision_score(y, ypred))
        rec.append(recall_score(y, ypred))
        f1.append(f1_score(y, ypred))
        fpr_value, tpr_value = fpr_tpr(y, ypred)
        fpr.append(fpr_value)
        tpr.append(tpr_value)
        qt.append(df.query_time.mean())
        qts.append(df.query_time.std())
#
    #print(f'{family.split(".")[0]:15}: acc:{np.mean(acc):0.2f}±{np.std(acc):.3f} f1:{np.mean(f1):0.2f}±{np.std(f1):.3f} pre:{np.mean(pre):0.2f}±{np.std(pre):.3f} rec:{np.mean(rec):0.2f}±{np.std(rec):.3f}  FPR:{np.mean(fpr):0.2f}±{np.std(fpr):.3f} TPR:{np.mean(tpr):0.2f}±{np.std(tpr):.3f} Query time: {np.mean(qt):0.5f}±{np.mean(qts):0.5f}')


    # Promedios por familia
    if acc:  # solo si hubo archivos válidos
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f} ')
              #f'Unknowns: {total_unknowns}')

        all_acc.append(np.mean(acc))
        all_pre.append(np.mean(pre))
        all_rec.append(np.mean(rec))
        all_f1.append(np.mean(f1))
        all_fpr.append(np.mean(fpr))
        all_tpr.append(np.mean(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.mean(qts))
        #total_unknowns_global += total_unknowns

# 🔍 Métricas globales
print("\n### 📊 Métricas globales ###")
print(f'Accuracy   : {np.mean(all_acc):.2f}')
print(f'F1-Score   : {np.mean(all_f1):.2f}')
print(f'Precision  : {np.mean(all_pre):.2f}')
print(f'Recall     : {np.mean(all_rec):.2f}')
print(f'FPR        : {np.mean(all_fpr):.2f}')
print(f'TPR        : {np.mean(all_tpr):.2f}')
print(f'Query time : {np.mean(all_qt):.5f} ± {np.mean(all_qts):.5f}')
#print(f'Total unknown classifications: {total_unknowns_global}')


alureon        : acc:0.97±0.018 f1:0.97±0.018 pre:0.97±0.019 rec:0.96±0.024 FPR:0.03±0.019 TPR:0.96±0.024 QT:0.13574±0.00454 
bamital        : acc:0.98±0.010 f1:0.98±0.009 pre:0.97±0.018 rec:1.00±0.000 FPR:0.03±0.019 TPR:1.00±0.000 QT:0.13996±0.00430 
banjori        : acc:0.97±0.016 f1:0.97±0.017 pre:0.97±0.019 rec:0.98±0.026 FPR:0.03±0.019 TPR:0.98±0.026 QT:0.13796±0.00918 
bedep          : acc:0.98±0.012 f1:0.98±0.012 pre:0.97±0.018 rec:0.99±0.012 FPR:0.03±0.019 TPR:0.99±0.012 QT:0.13863±0.00660 
charbot        : acc:0.79±0.043 f1:0.75±0.064 pre:0.95±0.032 rec:0.62±0.077 FPR:0.03±0.019 TPR:0.62±0.077 QT:0.14682±0.01002 
chinad         : acc:0.98±0.010 f1:0.98±0.009 pre:0.97±0.018 rec:1.00±0.000 FPR:0.03±0.019 TPR:1.00±0.000 QT:0.14399±0.00973 
conficker      : acc:0.91±0.031 f1:0.90±0.036 pre:0.97±0.022 rec:0.85±0.056 FPR:0.03±0.019 TPR:0.85±0.056 QT:0.14462±0.01920 
corebot        : acc:0.98±0.010 f1:0.98±0.009 pre:0.97±0.018 rec:1.00±0.000 FPR:0.03±0.019 TPR:1.00±0.000 QT:0.13730±0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


manuelita      : acc:0.51±0.018 f1:0.10±0.062 pre:0.59±0.244 rec:0.05±0.036 FPR:0.03±0.019 TPR:0.05±0.036 QT:0.14939±0.00471 
matsnu         : acc:0.95±0.015 f1:0.94±0.016 pre:0.97±0.019 rec:0.92±0.026 FPR:0.03±0.019 TPR:0.92±0.026 QT:0.14916±0.00318 
monerominer    : acc:0.98±0.010 f1:0.98±0.009 pre:0.97±0.018 rec:1.00±0.000 FPR:0.03±0.019 TPR:1.00±0.000 QT:0.14851±0.00354 
murofet        : acc:0.98±0.010 f1:0.98±0.010 pre:0.97±0.018 rec:1.00±0.005 FPR:0.03±0.019 TPR:1.00±0.005 QT:0.15070±0.00372 
murofetweekly  : acc:0.98±0.010 f1:0.98±0.009 pre:0.97±0.018 rec:1.00±0.000 FPR:0.03±0.019 TPR:1.00±0.000 QT:0.14812±0.00321 
mydoom         : acc:0.98±0.011 f1:0.98±0.010 pre:0.97±0.018 rec:1.00±0.007 FPR:0.03±0.019 TPR:1.00±0.007 QT:0.14813±0.00381 
necurs         : acc:0.97±0.015 f1:0.97±0.015 pre:0.97±0.019 rec:0.98±0.021 FPR:0.03±0.019 TPR:0.98±0.021 QT:0.14746±0.00344 
nymaim         : acc:0.83±0.031 f1:0.81±0.042 pre:0.96±0.026 rec:0.70±0.060 FPR:0.03±0.019 TPR:0.70±0.060 QT:0.14742±0

In [14]:
import requests
import pandas as pd
import numpy as np
import time
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import sys
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
import re

families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30

for family in families:
    print(family)
    dga = pd.read_csv(f'/content/drive/My Drive/New_Families/{family}', chunksize=50)
    legit = pd.read_csv('/content/drive/My Drive/Familias_Test/legit.gz', chunksize=50)
    dfs = []
    for run in range(runs):
        print(f'{run:2}/{runs}', end='\r')
        dfw = pd.concat([dga.get_chunk(), legit.get_chunk()])
        pred = []
        prob = []
        query_time = []
        results = []

        for domain_to_check in dfw.domain.values:
            st = time.time()

            result = predict_single_domain(loaded_model, domain_to_check)
            if result['prediction'] == "DGA":
                label_value = 1
            else:
                label_value = 0

            pred.append(label_value)
            query_time.append(time.time() - st)

        dfw['pred'] = pred
        # dfw['prob'] = prob  # Si tienes probabilidades, descomenta esta línea
        dfw['query_time'] = query_time
        dfw.to_csv(f'/content/drive/My Drive/results/results_Labin_{family}_{run}.csv.gz', index=False)


bazarbackdoor.gz
bazarbackdoor_v2.gz
bazarbackdoor_v3.gz
bigviktor.gz
bumblebee.gz
ccleaner.gz
dmsniff.gz
enviserv.gz
fosniw.gz
goz.gz
gozi_rfc4343.gz


KeyboardInterrupt: 

In [ ]:
#"""
families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]
#"""
def fpr_tpr(y, ypred):
    tn, fp, fn, tp = confusion_matrix(y, ypred).ravel()
    fpr = fp / (fp + tn)  # False Positive Rate
    tpr = tp / (tp + fn)  # True Positive Rate (Recall)
    return fpr, tpr

for family in families:
    acc = []
    pre = []
    rec = []
    f1 = []
    fpr = []
    tpr = []
    qt = []
    qts = []
    for run in range(runs):
        df = pd.read_csv(f'/content/drive/My Drive/results/results_Labin_{family}_{run}.csv.gz')
        y = (df.label == 'dga').astype(int)
        ypred = df.pred
        acc.append(accuracy_score(y, ypred))
        pre.append(precision_score(y, ypred))
        rec.append(recall_score(y, ypred))
        f1.append(f1_score(y, ypred))
        fpr_value, tpr_value = fpr_tpr(y, ypred)
        fpr.append(fpr_value)
        tpr.append(tpr_value)
        qt.append(df.query_time.mean())
        qts.append(df.query_time.std())
#
    #print(f'{family.split(".")[0]:15}: acc:{np.mean(acc):0.2f}±{np.std(acc):.3f} f1:{np.mean(f1):0.2f}±{np.std(f1):.3f} pre:{np.mean(pre):0.2f}±{np.std(pre):.3f} rec:{np.mean(rec):0.2f}±{np.std(rec):.3f}  FPR:{np.mean(fpr):0.2f}±{np.std(fpr):.3f} TPR:{np.mean(tpr):0.2f}±{np.std(tpr):.3f} Query time: {np.mean(qt):0.5f}±{np.mean(qts):0.5f}')


    # Promedios por familia
    if acc:  # solo si hubo archivos válidos
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f} '
              f'Unknowns: {total_unknowns}')

        all_acc.append(np.mean(acc))
        all_pre.append(np.mean(pre))
        all_rec.append(np.mean(rec))
        all_f1.append(np.mean(f1))
        all_fpr.append(np.mean(fpr))
        all_tpr.append(np.mean(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.mean(qts))
        total_unknowns_global += total_unknowns

# 🔍 Métricas globales
print("\n### 📊 Métricas globales ###")
print(f'Accuracy   : {np.mean(all_acc):.2f}')
print(f'F1-Score   : {np.mean(all_f1):.2f}')
print(f'Precision  : {np.mean(all_pre):.2f}')
print(f'Recall     : {np.mean(all_rec):.2f}')
print(f'FPR        : {np.mean(all_fpr):.2f}')
print(f'TPR        : {np.mean(all_tpr):.2f}')
print(f'Query time : {np.mean(all_qt):.5f} ± {np.mean(all_qts):.5f}')
print(f'Total unknown classifications: {total_unknowns_global}')
